In [ ]:
# 1. 強制解除安裝可能衝突的舊版 OpenCV
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python

# 2. 重新安裝所有必備套件的最穩定版本
!pip install psd-tools python-barcode pandas openpyxl opencv-python pillow

print("✅ 套件安裝完成！請點擊上方選單的「執行階段」->「重新啟動執行階段」。")

In [ ]:
import os
from psd_tools import PSDImage
from PIL import Image

print("📦 開始解壓縮照片...")
# 清理舊的資料夾並重新解壓
!rm -rf /content/photos
!unzip -q /content/5e.zip -d /content/photos/
print("✅ 照片解壓縮完成！")


In [ ]:
!pip install python-barcode pandas openpyxl pillow
# already installed at top

In [ ]:
!pip install mtcnn

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageEnhance # 🎯 新增了 ImageEnhance
from mtcnn import MTCNN
import barcode
from barcode.writer import ImageWriter
import os
import glob

In [ ]:


# ================= 1. 核心參數設定區 =================
CENTER_X = 324
FACE_CENTER_Y = 516
FACE_RADIUS = 170

# 🎯 照片美化設定 (推白程度)
PHOTO_BRIGHTNESS = 1.15  # 1.0 是原圖，1.15 代表調亮/推白 15%。如果覺得太亮或太暗，可以隨時微調這個數字！

# 🎯 姓名排版座標
ENG_ONLY_Y = 768
ENG_DUAL_Y = 742
CHI_DUAL_Y = 812

# 🎯 條碼與學號排版座標
BARCODE_W = 350
BARCODE_H = 90
BARCODE_BOTTOM_Y = 953
GAP_BARCODE_TEXT = 7
ID_TEXT_TOP_Y = BARCODE_BOTTOM_Y + GAP_BARCODE_TEXT

# 檔案與字體路徑
TEMPLATE_PATH = '/content/template.png'
EXCEL_PATH = '/content/2627studentlist_updated.xlsx'
FONT_ENG_PATH = '/content/ARLRDBD.TTF'
FONT_CHI_PATH = '/content/BiauKai.ttf'
FONT_ID_PATH = '/content/Arial.ttf'
PHOTO_DIR = '/content/photos'
OUTPUT_DIR = '/content/output/'

os.makedirs(OUTPUT_DIR, exist_ok=True)
detector = MTCNN()
# ===================================================

def process_face(photo_path):
    """使用 MTCNN 提取人臉，相片推白，並微調比例讓頭頂留白"""
    try:
        img_pil = Image.open(photo_path).convert("RGBA")

        # 🎯 關鍵新增：在這裡把相片推白/調亮
        enhancer = ImageEnhance.Brightness(img_pil)
        img_pil = enhancer.enhance(PHOTO_BRIGHTNESS)

        img_rgb = img_pil.convert("RGB")
        img_array = np.array(img_rgb)
    except Exception:
        return None

    faces = detector.detect_faces(img_array)
    orig_w, orig_h = img_pil.size

    if len(faces) == 0:
        nose_x, nose_y = orig_w // 2, orig_h // 2
        face_w_pixels = orig_w * 0.4
    else:
        box = faces[0]['box']
        keypoints = faces[0]['keypoints']
        nose_x, nose_y = keypoints['nose']
        face_w_pixels = box[2]

    # 把臉部佔比稍微降到 0.32，增加四周留白
    target_face_width = (FACE_RADIUS * 2) * 0.32
    scale_factor = target_face_width / face_w_pixels

    new_w, new_h = int(orig_w * scale_factor), int(orig_h * scale_factor)
    img_resized = img_pil.resize((new_w, new_h), Image.LANCZOS)

    new_nose_x = int(nose_x * scale_factor)
    new_nose_y = int(nose_y * scale_factor)

    # 把裁切框的 Y 軸往上移 (-10) 讓人像往下沉
    left = new_nose_x - FACE_RADIUS
    top = new_nose_y - FACE_RADIUS - 10
    right = left + (FACE_RADIUS * 2)
    bottom = top + (FACE_RADIUS * 2)

    return img_resized.crop((left, top, right, bottom))


def generate_id_card():
    print("🚀 開始批量生成學生證 (包含相片推白優化)...")

    try:
        bg_template = Image.open(TEMPLATE_PATH).convert("RGBA")
        true_mask = Image.open('/content/psd_mask.png').convert('L')
    except Exception as e:
        print(f"❌ 錯誤：找不到底圖或遮罩 ({e})")
        return

    try:
        df = pd.read_excel(EXCEL_PATH, dtype={'學號': str})
    except Exception as e:
        print(f"❌ 錯誤：讀取 Excel 失敗 ({e})")
        return

    try:
        font_eng = ImageFont.truetype(FONT_ENG_PATH, 46)
        font_chi = ImageFont.truetype(FONT_CHI_PATH, 65)
        font_id = ImageFont.truetype(FONT_ID_PATH, 42)
    except OSError:
        print(f"❌ 錯誤：找不到字體檔！請確認大小寫是否完全正確。")
        return

    success_count = 0

    for index, row in df.iterrows():
        student_id = str(row['學號']).strip()
        eng_name = str(row['Name']).strip()

        chi_name_raw = str(row['姓名']).strip()
        has_chi_name = (
            chi_name_raw != 'nan' and
            chi_name_raw != '' and
            chi_name_raw != '/' and
            chi_name_raw != '／'
        )

        photo_search = glob.glob(os.path.join(PHOTO_DIR, f"{student_id}.*"))
        if not photo_search:
            continue

        photo_path = photo_search[0]
        print(f"正在處理: {student_id} - {eng_name} ...", end=" ")

        bg = bg_template.copy()

        face_img = process_face(photo_path)
        if face_img:
            face_canvas = Image.new("RGBA", bg_template.size, (255, 255, 255, 255))
            face_canvas.paste(face_img, (155, 347), face_img)
            bg = Image.composite(face_canvas, bg, true_mask)

        draw = ImageDraw.Draw(bg)

        if has_chi_name:
            draw.text((CENTER_X, ENG_DUAL_Y), eng_name, font=font_eng, fill="black", anchor="mm")
            draw.text((CENTER_X, CHI_DUAL_Y), chi_name_raw, font=font_chi, fill="black", anchor="mm")
        else:
            draw.text((CENTER_X, ENG_ONLY_Y), eng_name, font=font_eng, fill="black", anchor="mm")

        barcode_options = {
            "write_text": False,
            "module_height": 10.0,
            "quiet_zone": 1.0,
            "background": "white"
        }
        Code128 = barcode.get_barcode_class('code128')
        code_img = Code128(student_id, writer=ImageWriter())

        temp_barcode_name = f"/content/temp_{student_id}"
        code_img.save(temp_barcode_name, options=barcode_options)

        bc_pil = Image.open(f"{temp_barcode_name}.png").convert("RGBA")
        bc_pil = bc_pil.resize((BARCODE_W, BARCODE_H), Image.NEAREST)

        bc_x = CENTER_X - (BARCODE_W // 2)
        bc_y = BARCODE_BOTTOM_Y - BARCODE_H
        bg.paste(bc_pil, (bc_x, bc_y), bc_pil)
        os.remove(f"{temp_barcode_name}.png")

        draw.text((CENTER_X, ID_TEXT_TOP_Y), student_id, font=font_id, fill="black", anchor="mt")

        output_path = os.path.join(OUTPUT_DIR, f"{student_id}_ID.png")
        bg.save(output_path)
        print("完成！")
        success_count += 1

    print(f"\n✅ 任務結束！成功生成 {success_count} 張完美排版的學生證。")

generate_id_card()

In [ ]:
from google.colab import files

# 1. 將 output 資料夾靜默打包成 output.zip (-r 代表遞迴，-q 代表靜默不印出明細)
!zip -r -q output.zip /content/output

# 2. 觸發瀏覽器下載檔案
files.download('output.zip')

In [ ]:
# 徹底刪除舊資料夾，避免混入舊的學生證
!rm -rf /content/photos
!rm -rf /content/output
!rm -rf /content/output_pdfs

In [ ]:
!rm -rf /content/output
!rm -rf /content/output_pdfs

In [ ]:
# 1. 安裝所需套件 (合併 PDF 專用)
!pip install -q img2pdf

import os
import img2pdf
import shutil
from google.colab import files

In [ ]:


# batch combine face and back 2 pdf
# ================= 路徑設定區 =================
# 讀取剛剛生成的學生證正面資料夾
input_folder = "/content/output"
# 學生證背面的統一底圖 (請確認已上傳)
fixed_png = "/content/back2026.png"
# 存放合併後 PDF 的資料夾
output_folder = "/content/output_pdfs"
# ============================================

# 清空舊資料夾並建立新的，避免重複執行混入舊檔
!rm -rf /content/output_pdfs
os.makedirs(output_folder, exist_ok=True)

# 確保背面底圖存在
if not os.path.exists(fixed_png):
    print(f"❌ 錯誤：找不到背面底圖 '{fixed_png}'，請確認是否已上傳到左側目錄。")
else:
    # 2. 搜尋 /content/output/ 下的所有學生證正面圖片
    png_files = []
    for root, dirs, files_in_dir in os.walk(input_folder):
        # 避開 Mac 的 __MACOSX 隱藏資料夾
        if '__MACOSX' in root:
            continue
        for file in files_in_dir:
            if file.lower().endswith('.png') and not file.startswith('._'):
                png_files.append(os.path.join(root, file))

    if not png_files:
        print(f"⚠️ 在 {input_folder} 找不到任何圖片，請確認上一階段的生成學生證程式已成功執行。")
    else:
        # 🌟 關鍵修改 1：對檔案路徑進行排序，確保依照檔名 (學號) 從小到大排列
        png_files.sort(key=lambda x: os.path.basename(x))

        print(f"🔍 找到 {len(png_files)} 張學生證正面，開始合併為雙面 PDF...")

        # 🌟 關鍵修改 2：建立一個大陣列，用來裝所有要放進「總檔 PDF」的圖片頁面
        all_pages_combined = []

        for img_path in png_files:
            base_name = os.path.splitext(os.path.basename(img_path))[0]
            output_pdf_path = os.path.join(output_folder, f"{base_name}.pdf")

            # 頁面順序：第一頁為正面(帶照片條碼)，第二頁為統一背面
            pages = [img_path, fixed_png]

            # 寫入獨立的 PDF
            with open(output_pdf_path, "wb") as f:
                f.write(img2pdf.convert(pages))

            # 將這兩頁依序加入總陣列中，為了稍後的總檔做準備
            all_pages_combined.extend(pages)

        print(f"✅ 獨立 PDF 合併完成！共生成 {len(png_files)} 個。")

        # 🌟 關鍵修改 3：利用收集好的所有頁面，一次性生成一個「總檔 PDF」
        # 檔名開頭加上 00_，這樣解開 ZIP 後它會置頂，方便尋找
        print("📄 正在生成全班總檔 PDF，請稍候...")
        combined_pdf_path = os.path.join(output_folder, "All_Students_Combined.pdf")

        with open(combined_pdf_path, "wb") as f:
            f.write(img2pdf.convert(all_pages_combined))

        print(f"✅ 總檔 PDF 生成完成！包含所有學生的正反面，共 {len(all_pages_combined)} 頁。")

        # 4. 打包成 ZIP 並自動觸發下載 (總檔已經在 output_folder 裡，會自動一起被打包)
        print("📦 正在打包 ZIP，準備下載到您的電腦...")
        zip_filename = '/content/1016_student_cards_pdf'
        shutil.make_archive(zip_filename, 'zip', output_folder)
        files.download(f'{zip_filename}.zip')

In [ ]:
files.download(f'{zip_filename}.zip')

In [ ]:
----Below crop for eclass----

In [ ]:
import os
from psd_tools import PSDImage
from PIL import Image

In [ ]:


print("📦 開始解壓縮照片...")
# 清理舊的資料夾並重新解壓
!rm -rf /content/photos
!unzip -q /content/5m.zip -d /content/photos/
print("✅ 照片解壓縮完成！")


In [ ]:
import os
import numpy as np
from PIL import Image
from mtcnn import MTCNN
import gc

# ================= 1. 參數設定區 =================
INPUT_DIR = '/content/photos'           # 🌟 您的原始照片資料夾
OUTPUT_DIR = '/content/1016_cropped'   # 🌟 裁剪完存放的新資料夾

TARGET_RATIO = 10 / 13.0                # 目標比例 (寬 : 高 = 10 : 13)
# =================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
detector = MTCNN()

print(f"🚀 開始執行 10:13 智慧裁剪，目標：最大化保留上半身...")
processed_count = 0

for root, dirs, files_in_dir in os.walk(INPUT_DIR):
    for file_name in files_in_dir:
        if file_name.lower().endswith(('.png', '.jpg', '.jpeg')) and not file_name.startswith('._'):
            input_path = os.path.join(root, file_name)
            output_path = os.path.join(OUTPUT_DIR, file_name)

            try:
                img = Image.open(input_path).convert("RGB")
                W, H = img.size
                current_ratio = W / float(H)

                # 🛠️ 步驟 A：用 MTCNN 找臉，用來當作裁剪的「錨點」
                img_array = np.array(img)
                faces = detector.detect_faces(img_array)

                if len(faces) > 0:
                    box = faces[0]['box']
                    face_x, face_y, face_w, face_h = box
                    face_center_x = face_x + face_w // 2
                    face_center_y = face_y + face_h // 2
                else:
                    # 如果萬一找不到臉 (防呆)，就預設臉在畫面上方 40% 的正中間
                    face_center_x = W // 2
                    face_center_y = int(H * 0.4)

                # 🛠️ 步驟 B：計算最大可能保留的 10:13 裁切框
                if current_ratio > TARGET_RATIO:
                    # 原圖「太寬」：保留全部高度，裁切左右
                    new_h = H
                    new_w = int(H * TARGET_RATIO)

                    # 臉部保持在水平正中間
                    left = face_center_x - (new_w // 2)
                    top = 0

                else:
                    # 原圖「太長」：保留全部寬度，裁切上下
                    new_w = W
                    new_h = int(W / TARGET_RATIO)

                    # 🌟 關鍵排版：確保臉部中心落在新畫面的「上方 35%」
                    # 這樣頭頂能保留適當空間，下方則能盡可能留下最多的肩膀與身體
                    top = face_center_y - int(new_h * 0.35)
                    left = 0

                # 🛠️ 步驟 C：邊界防呆機制 (防止裁切框超出原圖範圍)
                if left < 0:
                    left = 0
                elif left + new_w > W:
                    left = W - new_w

                if top < 0:
                    top = 0
                elif top + new_h > H:
                    top = H - new_h

                right = left + new_w
                bottom = top + new_h

                # 🛠️ 步驟 D：執行裁剪並儲存
                cropped_img = img.crop((left, top, right, bottom))
                cropped_img.save(output_path, quality=95)

                print(f"✅ 成功裁剪: {file_name}")
                processed_count += 1

                # 清理記憶體
                del img, cropped_img, img_array
                gc.collect()

            except Exception as e:
                print(f"❌ 處理 {file_name} 發生錯誤: {e}")

print(f"\n🎉 任務結束！共完美裁剪了 {processed_count} 張照片。")
print(f"📁 檔案已存放在: {OUTPUT_DIR}")

In [ ]:
from google.colab import files

# 1. 加入 -j 參數，並在路徑最後加上 /*
# -j: 丟棄所有外層資料夾結構，只打包檔案本身
# -q: 靜默執行
!zip -j -q 1016_eclass.zip /content/1016_cropped/*

# 2. 觸發瀏覽器下載檔案
files.download('1016_eclass.zip')

In [ ]:
!rm -rf /content/photos
!rm -rf /content/output
!rm -rf /content/output_pdfs